# Chapter 2 · Module 3 — PyTorch Data & Training Pipeline on Flowers-102

**Goal:** Build the full PyTorch training pipeline from scratch on the Oxford **Flowers-102** dataset:

1. A **Custom `Dataset`** class that parses the raw Oxford files directly (no `torchvision.datasets.Flowers102` shortcut) — `.mat` label/split files + a folder of `.jpg` images.
2. A **`DataLoader`** with training-time augmentation and validation-time deterministic transforms.
3. A simple **CNN** (following Module 1's CNN architecture concepts) trained with PyTorch's autograd.
4. A **training loop** and a **validation loop**.
5. **Early stopping** based on validation loss.
6. **Checkpoint saving** (best + last) and **checkpoint loading** (resume training / load-best-for-inference).

> **Runtime:** Runtime → Change runtime type → **T4 GPU** in Google Colab.

**Dataset:** [Oxford 102 Category Flower Dataset](https://www.robots.ox.ac.uk/~vgg/data/flowers/102/) — 102 flower categories common in the UK, ~8,189 images total. We download the raw files ourselves:
- `102flowers.tgz` — all JPEG images, flat folder, 1-indexed filenames (`image_00001.jpg` ...)
- `imagelabels.mat` — 1×8189 array of class labels (1–102) aligned to filename order
- `setid.mat` — `trnid`, `valid`, `tstid` arrays of 1-indexed image ids defining the official split


## 1. Environment setup
Check we have a GPU, then install/import what we need. `scipy` is used to read the Oxford `.mat` files — it ships with Colab already.

In [ ]:
import torch
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DEVICE

In [ ]:
import os, io, time, tarfile, random, copy
from pathlib import Path

import numpy as np
import scipy.io
import requests
from tqdm.auto import tqdm

from PIL import Image
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

## 2. Download the raw Oxford Flowers-102 files

We fetch the three files straight from the Oxford VGG server and extract the images tarball. This gives us the *raw* file layout — a flat folder of images plus two `.mat` files — exactly what we'd have to deal with if `torchvision` didn't provide a wrapper for this dataset.

In [ ]:
DATA_DIR = Path("./data/flowers-102")
IMAGES_DIR = DATA_DIR / "jpg"
DATA_DIR.mkdir(parents=True, exist_ok=True)

BASE_URL = "https://www.robots.ox.ac.uk/~vgg/data/flowers/102"
FILES = {
    "102flowers.tgz": f"{BASE_URL}/102flowers.tgz",
    "imagelabels.mat": f"{BASE_URL}/imagelabels.mat",
    "setid.mat": f"{BASE_URL}/setid.mat",
}

def download(url: str, dest: Path, chunk_size: int = 1 << 16):
    if dest.exists():
        print(f"Already downloaded: {dest.name}")
        return
    resp = requests.get(url, stream=True, timeout=60)
    resp.raise_for_status()
    total = int(resp.headers.get("content-length", 0))
    with open(dest, "wb") as f, tqdm(total=total, unit="B", unit_scale=True, desc=dest.name) as bar:
        for chunk in resp.iter_content(chunk_size=chunk_size):
            f.write(chunk)
            bar.update(len(chunk))

for fname, url in FILES.items():
    download(url, DATA_DIR / fname)

if not IMAGES_DIR.exists():
    print("Extracting images...")
    with tarfile.open(DATA_DIR / "102flowers.tgz") as tar:
        tar.extractall(DATA_DIR)
    print("Done.")
else:
    print("Images already extracted.")

n_images = len(list(IMAGES_DIR.glob("*.jpg")))
print(f"Found {n_images} images in {IMAGES_DIR}")

> **Fallback (if the Oxford server is slow/unreachable from Colab):** uncomment the cell below to fall back to `torchvision.datasets.Flowers102`, which mirrors the same files from a different host. The custom `Dataset` class in Section 3 only needs `IMAGES_DIR`, `imagelabels.mat` and `setid.mat` to exist at the paths above, so either source works.

In [ ]:
# from torchvision.datasets import Flowers102
# for split in ["train", "val", "test"]:
#     Flowers102(root=str(DATA_DIR.parent), split=split, download=True)
# # torchvision extracts to ./data/flowers-102/flowers-102/{jpg,imagelabels.mat,setid.mat}

## 3. Custom `Dataset` class

We parse the raw files ourselves instead of relying on a pre-built dataset wrapper:

- `imagelabels.mat` → `scipy.io.loadmat` gives a `'labels'` array, 1×8189, **1-indexed classes (1–102)**. Index `i` (0-based) corresponds to `image_{i+1:05d}.jpg`.
- `setid.mat` → `'trnid'`, `'valid'`, `'tstid'`, each a 1-indexed array of **image ids** (not 0-based positions) belonging to that split.
- Images are loaded lazily in `__getitem__` (not all held in memory), and `PIL` + `torchvision.transforms` handle decoding/augmentation.

Labels are converted to 0-indexed (`0–101`) for use with `nn.CrossEntropyLoss`.

In [ ]:
class Flowers102RawDataset(Dataset):
    """Parses the raw Oxford Flowers-102 files (images dir + imagelabels.mat + setid.mat)."""

    SPLIT_KEYS = {"train": "trnid", "val": "valid", "test": "tstid"}

    def __init__(self, data_dir, split="train", transform=None):
        assert split in self.SPLIT_KEYS, f"split must be one of {list(self.SPLIT_KEYS)}"
        self.data_dir = Path(data_dir)
        self.images_dir = self.data_dir / "jpg"
        self.transform = transform

        labels_mat = scipy.io.loadmat(self.data_dir / "imagelabels.mat")
        all_labels = labels_mat["labels"].squeeze()  # 1-indexed, length 8189, position i -> image_{i+1}

        setid_mat = scipy.io.loadmat(self.data_dir / "setid.mat")
        image_ids = setid_mat[self.SPLIT_KEYS[split]].squeeze()  # 1-indexed image ids for this split

        self.samples = []
        for image_id in sorted(image_ids.tolist()):
            filename = f"image_{image_id:05d}.jpg"
            label_0indexed = int(all_labels[image_id - 1]) - 1  # position i -> image_id i+1
            self.samples.append((filename, label_0indexed))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        filename, label = self.samples[idx]
        image = Image.open(self.images_dir / filename).convert("RGB")
        if self.transform is not None:
            image = self.transform(image)
        return image, label

    @property
    def num_classes(self):
        return 102

In [ ]:
# Sanity check without transforms first
raw_train = Flowers102RawDataset(DATA_DIR, split="train")
raw_val = Flowers102RawDataset(DATA_DIR, split="val")
raw_test = Flowers102RawDataset(DATA_DIR, split="test")

print(f"train: {len(raw_train)}  val: {len(raw_val)}  test: {len(raw_test)}")

img, label = raw_train[0]
print("Sample image size:", img.size, "| label:", label)

## 4. Transforms & `DataLoader`

Training gets light augmentation (random crop/flip/color jitter); validation and test use a deterministic resize+center-crop so numbers are comparable run to run. Normalization uses standard ImageNet statistics — reasonable defaults even though we're training the CNN from scratch.

In [ ]:
IMG_SIZE = 128
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

from torchvision import transforms as T

train_transform = T.Compose([
    T.RandomResizedCrop(IMG_SIZE, scale=(0.7, 1.0)),
    T.RandomHorizontalFlip(),
    T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    T.ToTensor(),
    T.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

eval_transform = T.Compose([
    T.Resize(int(IMG_SIZE * 1.15)),
    T.CenterCrop(IMG_SIZE),
    T.ToTensor(),
    T.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

train_dataset = Flowers102RawDataset(DATA_DIR, split="train", transform=train_transform)
val_dataset = Flowers102RawDataset(DATA_DIR, split="val", transform=eval_transform)
test_dataset = Flowers102RawDataset(DATA_DIR, split="test", transform=eval_transform)

BATCH_SIZE = 64
NUM_WORKERS = 2  # Colab: 2 is usually safe; bump if you have more CPU cores

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                           num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False,
                         num_workers=NUM_WORKERS, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True)

print(f"train batches: {len(train_loader)} | val batches: {len(val_loader)} | test batches: {len(test_loader)}")

In [ ]:
# Visualize a batch to make sure images/labels line up
def denormalize(tensor):
    mean = torch.tensor(IMAGENET_MEAN).view(3, 1, 1)
    std = torch.tensor(IMAGENET_STD).view(3, 1, 1)
    return (tensor * std + mean).clamp(0, 1)

images, labels = next(iter(train_loader))
fig, axes = plt.subplots(2, 6, figsize=(16, 6))
for i, ax in enumerate(axes.flat):
    ax.imshow(denormalize(images[i]).permute(1, 2, 0).numpy())
    ax.set_title(f"class {labels[i].item()}")
    ax.axis("off")
plt.tight_layout()
plt.show()

## 5. Model — a simple CNN

A straightforward stack of conv blocks (Conv → BatchNorm → ReLU → MaxPool) followed by a classifier head, in the spirit of the basic CNN architecture from Module 1. Trained from scratch on 102 classes with relatively little data — expect it to plateau well below a transfer-learning approach, which is fine for practicing the *pipeline* mechanics.

In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self, num_classes=102, img_size=128):
        super().__init__()

        def conv_block(in_ch, out_ch):
            return nn.Sequential(
                nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1),
                nn.BatchNorm2d(out_ch),
                nn.ReLU(inplace=True),
                nn.MaxPool2d(2),
            )

        self.features = nn.Sequential(
            conv_block(3, 32),
            conv_block(32, 64),
            conv_block(64, 128),
            conv_block(128, 256),
        )
        reduced = img_size // (2 ** 4)
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Dropout(0.4),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        return self.classifier(x)


model = SimpleCNN(num_classes=train_dataset.num_classes, img_size=IMG_SIZE).to(DEVICE)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable parameters: {n_params:,}")

## 6. Early stopping & checkpointing utilities

- `EarlyStopping` watches validation loss and signals when to stop after `patience` epochs without improvement.
- `save_checkpoint` / `load_checkpoint` persist (and restore) model weights, optimizer state, epoch number, and best validation loss — enough to fully resume training, not just reload weights for inference.

In [ ]:
class EarlyStopping:
    def __init__(self, patience=7, min_delta=0.0):
        self.patience = patience
        self.min_delta = min_delta
        self.best_loss = float("inf")
        self.counter = 0
        self.should_stop = False

    def step(self, val_loss):
        if val_loss < self.best_loss - self.min_delta:
            self.best_loss = val_loss
            self.counter = 0
            return True  # improved
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.should_stop = True
            return False  # no improvement

In [ ]:
CHECKPOINT_DIR = Path("./checkpoints")
CHECKPOINT_DIR.mkdir(exist_ok=True)
LAST_CKPT = CHECKPOINT_DIR / "last.pt"
BEST_CKPT = CHECKPOINT_DIR / "best.pt"


def save_checkpoint(path, model, optimizer, epoch, best_val_loss, history):
    torch.save({
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "best_val_loss": best_val_loss,
        "history": history,
    }, path)


def load_checkpoint(path, model, optimizer=None, map_location=None):
    checkpoint = torch.load(path, map_location=map_location or DEVICE)
    model.load_state_dict(checkpoint["model_state_dict"])
    if optimizer is not None and "optimizer_state_dict" in checkpoint:
        optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
    return checkpoint  # caller can read epoch / best_val_loss / history back out

## 7. Training loop & validation loop

Two small functions — `train_one_epoch` and `evaluate` — each looping over their respective `DataLoader`. Keeping them separate mirrors how you'd structure this outside a notebook (e.g. in a `train.py` / `engine.py`).

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss, running_correct, n_samples = 0.0, 0, 0

    pbar = tqdm(loader, desc="train", leave=False)
    for images, labels in pbar:
        images, labels = images.to(device, non_blocking=True), labels.to(device, non_blocking=True)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        batch_size = images.size(0)
        running_loss += loss.item() * batch_size
        running_correct += (outputs.argmax(1) == labels).sum().item()
        n_samples += batch_size
        pbar.set_postfix(loss=running_loss / n_samples, acc=running_correct / n_samples)

    return running_loss / n_samples, running_correct / n_samples


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    running_loss, running_correct, n_samples = 0.0, 0, 0

    pbar = tqdm(loader, desc="val", leave=False)
    for images, labels in pbar:
        images, labels = images.to(device, non_blocking=True), labels.to(device, non_blocking=True)

        outputs = model(images)
        loss = criterion(outputs, labels)

        batch_size = images.size(0)
        running_loss += loss.item() * batch_size
        running_correct += (outputs.argmax(1) == labels).sum().item()
        n_samples += batch_size
        pbar.set_postfix(loss=running_loss / n_samples, acc=running_correct / n_samples)

    return running_loss / n_samples, running_correct / n_samples

## 8. Run training

Wires everything together: optimizer, LR scheduler, early stopping, and checkpointing. `best.pt` is only overwritten when validation loss improves; `last.pt` is written every epoch so you can resume a crashed/disconnected Colab session from where it left off.

In [ ]:
NUM_EPOCHS = 40
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
PATIENCE = 8

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=3)
early_stopping = EarlyStopping(patience=PATIENCE, min_delta=1e-4)

history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
start_epoch = 0
best_val_loss = float("inf")

# Resume from a previous run, if a checkpoint already exists (comment out to always start fresh)
if LAST_CKPT.exists():
    ckpt = load_checkpoint(LAST_CKPT, model, optimizer)
    start_epoch = ckpt["epoch"] + 1
    best_val_loss = ckpt["best_val_loss"]
    history = ckpt["history"]
    early_stopping.best_loss = best_val_loss
    print(f"Resumed from checkpoint at epoch {start_epoch}, best_val_loss={best_val_loss:.4f}")

In [ ]:
for epoch in range(start_epoch, NUM_EPOCHS):
    t0 = time.time()

    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, DEVICE)
    val_loss, val_acc = evaluate(model, val_loader, criterion, DEVICE)
    scheduler.step(val_loss)

    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)

    improved = early_stopping.step(val_loss)
    if improved:
        best_val_loss = val_loss
        save_checkpoint(BEST_CKPT, model, optimizer, epoch, best_val_loss, history)

    save_checkpoint(LAST_CKPT, model, optimizer, epoch, best_val_loss, history)

    dt = time.time() - t0
    marker = " *" if improved else ""
    print(f"Epoch {epoch+1:3d}/{NUM_EPOCHS} | "
          f"train_loss {train_loss:.4f} acc {train_acc:.3f} | "
          f"val_loss {val_loss:.4f} acc {val_acc:.3f} | "
          f"{dt:.1f}s{marker}")

    if early_stopping.should_stop:
        print(f"Early stopping triggered after {epoch+1} epochs "
              f"(no val_loss improvement for {PATIENCE} epochs).")
        break

## 9. Training curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

axes[0].plot(history["train_loss"], label="train")
axes[0].plot(history["val_loss"], label="val")
axes[0].set_title("Loss")
axes[0].set_xlabel("epoch")
axes[0].legend()

axes[1].plot(history["train_acc"], label="train")
axes[1].plot(history["val_acc"], label="val")
axes[1].set_title("Accuracy")
axes[1].set_xlabel("epoch")
axes[1].legend()

plt.tight_layout()
plt.show()

## 10. Load best checkpoint & evaluate on the held-out test set

Demonstrates loading a checkpoint into a **fresh model instance** (as you'd do in a separate inference script), then running the validation loop's `evaluate` function against `test_loader`.

In [ ]:
inference_model = SimpleCNN(num_classes=test_dataset.num_classes, img_size=IMG_SIZE).to(DEVICE)
ckpt = load_checkpoint(BEST_CKPT, inference_model)  # optimizer=None: weights only, for inference
print(f"Loaded best checkpoint from epoch {ckpt['epoch']+1} (val_loss={ckpt['best_val_loss']:.4f})")

test_loss, test_acc = evaluate(inference_model, test_loader, criterion, DEVICE)
print(f"Test loss: {test_loss:.4f} | Test accuracy: {test_acc:.3f}")

In [ ]:
# Visualize predictions on a batch of test images
inference_model.eval()
images, labels = next(iter(test_loader))
with torch.no_grad():
    preds = inference_model(images.to(DEVICE)).argmax(1).cpu()

fig, axes = plt.subplots(2, 6, figsize=(16, 6))
for i, ax in enumerate(axes.flat):
    ax.imshow(denormalize(images[i]).permute(1, 2, 0).numpy())
    correct = preds[i].item() == labels[i].item()
    ax.set_title(f"pred {preds[i].item()} / true {labels[i].item()}",
                 color="green" if correct else "red")
    ax.axis("off")
plt.tight_layout()
plt.show()

## Recap

- **Custom `Dataset`** (`Flowers102RawDataset`): parses `imagelabels.mat` + `setid.mat` + a flat image folder directly, no `torchvision.datasets.Flowers102` wrapper.
- **`DataLoader`**: separate train (augmented) / val / test (deterministic) pipelines.
- **Training loop** / **validation loop**: factored into `train_one_epoch` / `evaluate`, called once per epoch.
- **Early stopping**: `EarlyStopping` class watching validation loss with a patience window.
- **Checkpointing**: `save_checkpoint` writes `last.pt` every epoch (resume-safe) and `best.pt` on improvement; `load_checkpoint` restores model (+ optimizer, epoch, history) from either.

**Ideas to extend:**
- Swap `SimpleCNN` for a pretrained backbone (`torchvision.models.resnet18(weights=...)`, fine-tuned) and compare accuracy/convergence speed.
- Add mixed-precision training (`torch.cuda.amp`) to speed up T4 training.
- Try `torch.optim.lr_scheduler.CosineAnnealingLR` instead of `ReduceLROnPlateau`.
- Log metrics to TensorBoard or Weights & Biases instead of matplotlib.
